# Fertilizer Recommendation System
## A Predictive Analytics Approach Using Machine Learning

**Author:** Alejandro Barea  
**Course:** AI for Business Analytics — Purdue University  
**Framework:** Problem – Data – Insights – Deployment (PDID)

---

## 1. Business Understanding

### 1.1 Problem Definition

Fertilizer selection is one of the most critical decisions in modern agriculture. Applying the wrong type or amount of fertilizer can lead to poor crop yields, nutrient imbalances in the soil, environmental degradation through nutrient runoff, and significant financial losses for farmers.

Traditionally, fertilizer recommendations depend on the expertise of agronomists who analyze soil tests, environmental conditions, and crop requirements. However, this process is time-consuming, inconsistent, and not always accessible to smallholder farmers.

### 1.2 Objective

The goal of this project is to develop a **machine learning classification model** that recommends the optimal fertilizer based on:

- **Soil characteristics** (type, pH, moisture, organic carbon, electrical conductivity)
- **Nutrient levels** (Nitrogen, Phosphorus, Potassium)
- **Environmental conditions** (temperature, humidity, rainfall)
- **Agricultural context** (crop type, growth stage, season, irrigation, region)

### 1.3 Research Questions

1. Which soil and environmental variables most influence fertilizer recommendation?
2. Can machine learning models accurately predict the optimal fertilizer?
3. Which features are the most important drivers of fertilizer recommendation?

### 1.4 Business Value

A reliable recommendation system can help farmers make data-driven decisions, reduce waste, optimize yields, and contribute to sustainable farming practices. For agricultural companies (such as Fertinagro Biotech), such a system can support advisory services and precision agriculture products.

## 2. Environment Setup

In [ ]:
# Install required packages (run this cell once, then restart the kernel if needed)
import subprocess, sys

packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'joblib']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All packages installed successfully. If this is the first run, restart the kernel and run all cells again.')

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import joblib

# Plot style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

print('All libraries loaded successfully.')

## 3. Data Loading and Inspection

We begin by loading the dataset and understanding its structure. The CSV uses a semicolon (`;`) as separator.

In [ ]:
# Load dataset — semicolon-separated
df = pd.read_csv('fertilizer_recommendation.csv', sep=';')
print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Column names and data types
df.info()

In [ ]:
# Summary statistics for numerical variables
df.describe().T

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')

# Check for duplicates
print(f'\nDuplicate rows: {df.duplicated().sum()}')

In [ ]:
# Identify variable types
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include='number').columns.tolist()

# The target variable
target = 'Recommended_Fertilizer'

# Remove target from feature lists
cat_features = [c for c in cat_cols if c != target]
num_features = num_cols.copy()

print(f'Target variable: {target}')
print(f'Target classes ({df[target].nunique()}): {list(df[target].unique())}')
print(f'\nCategorical features ({len(cat_features)}): {cat_features}')
print(f'Numerical features ({len(num_features)}): {num_features}')

### Data Inspection Summary

The dataset contains **10,000 observations** and **20 variables** (19 features + 1 target). There are **no missing values** and **no duplicate rows**, which means the data is clean and ready for analysis. The target variable `Recommended_Fertilizer` has **7 classes**: MOP, Urea, Zinc Sulphate, Compost, NPK, DAP, and SSP.

There are **7 categorical features** (Soil_Type, Crop_Type, Crop_Growth_Stage, Season, Irrigation_Type, Previous_Crop, Region) and **12 numerical features** covering soil properties, nutrient levels, environmental conditions, and historical yield data.

## 4. Exploratory Data Analysis (EDA)

### 4.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
order = df[target].value_counts().index
sns.countplot(y=target, data=df, order=order, ax=axes[0])
axes[0].set_title('Fertilizer Recommendation — Frequency')
axes[0].set_xlabel('Count')

# Pie chart
df[target].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[1],
                                    colors=sns.color_palette('Set2', df[target].nunique()))
axes[1].set_ylabel('')
axes[1].set_title('Fertilizer Recommendation — Proportion')

plt.tight_layout()
plt.show()

print(df[target].value_counts())

### 4.2 Distribution of Key Numerical Variables

In [ ]:
# Select key numerical features for visualization
key_nums = ['Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level',
            'Soil_pH', 'Soil_Moisture', 'Temperature', 'Humidity', 'Rainfall']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(key_nums):
    ax = axes[i // 4, i % 4]
    sns.histplot(df[col], kde=True, ax=ax, color=sns.color_palette('Set2')[i % 8])
    ax.set_title(col)
plt.suptitle('Distribution of Key Numerical Variables', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 4.3 Nutrient Levels by Fertilizer Type

In [ ]:
nutrients = ['Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, nut in enumerate(nutrients):
    sns.boxplot(x=target, y=nut, data=df, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{nut} by Fertilizer Type')
    axes[i].tick_params(axis='x', rotation=45)
plt.suptitle('Nutrient Levels Across Fertilizer Recommendations', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 4.4 Environmental Variables by Fertilizer Type

In [ ]:
env_vars = ['Temperature', 'Humidity', 'Rainfall', 'Soil_pH']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, var in enumerate(env_vars):
    sns.boxplot(x=target, y=var, data=df, ax=axes[i], palette='Set3')
    axes[i].set_title(f'{var} by Fertilizer')
    axes[i].tick_params(axis='x', rotation=45)
plt.suptitle('Environmental Variables Across Fertilizer Recommendations', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 4.5 Categorical Variable Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
cat_plot_vars = ['Soil_Type', 'Crop_Type', 'Season', 'Irrigation_Type', 'Region', 'Crop_Growth_Stage']

for i, col in enumerate(cat_plot_vars):
    ax = axes[i // 3, i % 3]
    ct = pd.crosstab(df[col], df[target], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
    ax.set_title(f'Fertilizer Distribution by {col}')
    ax.set_ylabel('Percentage')
    ax.legend(fontsize=7, loc='upper right')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Fertilizer Recommendation Patterns Across Categorical Variables', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 4.6 Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 9))
corr = df[num_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, square=True)
plt.title('Correlation Matrix — Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

### 4.7 Key EDA Insights

Based on the exploratory analysis, we can draw the following conclusions:

1. **Balanced target distribution:** The 7 fertilizer classes are roughly balanced (each around 13–15% of observations), which means we do not need to apply class balancing techniques. This is ideal for training classification models without bias toward majority classes.

2. **Nutrient levels show variation across fertilizer types:** Boxplots reveal that Nitrogen, Phosphorus, and Potassium levels differ across fertilizer recommendations, suggesting they carry predictive signal. For example, higher potassium levels may be associated with MOP (Muriate of Potash), which aligns with agronomic knowledge.

3. **Environmental conditions are fairly uniform:** Temperature, humidity, and rainfall distributions show overlap across fertilizer types, indicating that while they contribute context, they may not be the strongest individual predictors. The model will need to capture interactions between these variables.

4. **Low multicollinearity among numerical features:** The correlation heatmap shows no strong pairwise correlations among the numerical features, which is favorable for model training — each variable contributes independent information.

5. **Categorical variables add important context:** The stacked bar charts show that certain soil types, crop types, and regions have noticeably different fertilizer distributions. For instance, crop type and growth stage appear to influence which fertilizer is recommended, which aligns with real-world agricultural practice.

## 5. Data Preprocessing

In this section we prepare the data for modeling: encode categorical variables, separate features and target, split into training and test sets, and apply scaling for distance-based models.

In [ ]:
# Create a copy for preprocessing
data = df.copy()

# Encode categorical features using LabelEncoder
label_encoders = {}
for col in cat_features:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le
    print(f'{col}: {list(le.classes_)}')

# Encode target variable
le_target = LabelEncoder()
data[target] = le_target.fit_transform(data[target])
print(f'\nTarget ({target}): {list(le_target.classes_)}')

In [ ]:
# Define features (X) and target (y)
feature_cols = cat_features + num_features
X = data[feature_cols]
y = data[target]

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape: {y.shape}')
print(f'\nFeatures used ({len(feature_cols)}):')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i}. {col}')

In [ ]:
# Train-test split (80/20) with stratification to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'\nClass distribution in training set:')
print(y_train.value_counts().sort_index().rename(index=dict(enumerate(le_target.classes_))))

In [ ]:
# Apply StandardScaler — fitted on training data ONLY to prevent data leakage
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)

print('Scaling applied. Mean of training features (should be ~0):')
print(X_train_scaled.mean().round(6).head())

### Preprocessing Summary

- **Categorical encoding:** All 7 categorical features were encoded using `LabelEncoder`. The encoders are saved for reuse during deployment.
- **Train/test split:** 80% training (8,000 samples) and 20% test (2,000 samples), stratified by the target variable to maintain class proportions.
- **Scaling:** `StandardScaler` was fitted **only on training data** and then applied to both training and test sets. This prevents data leakage and ensures the model generalizes properly.
- **No missing values** were found, so no imputation was necessary.

## 6. Model Development and Evaluation

We train three classification models and evaluate each using standard metrics. Tree-based models (Decision Tree, Random Forest) use unscaled data; KNN uses scaled data since it is distance-based.

In [ ]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name, class_names):
    """Train a model, evaluate it, and return a results dictionary."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='weighted')
    rec = recall_score(y_te, y_pred, average='weighted')
    f1 = f1_score(y_te, y_pred, average='weighted')
    
    print(f'=== {model_name} ===')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1-score:  {f1:.4f}')
    print(f'\nClassification Report:\n')
    print(classification_report(y_te, y_pred, target_names=class_names))
    
    # Confusion Matrix
    fig, ax = plt.subplots(figsize=(8, 6))
    cm = confusion_matrix(y_te, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(f'Confusion Matrix — {model_name}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    return {'Model': model_name, 'Accuracy': acc, 'Precision': prec,
            'Recall': rec, 'F1-Score': f1, 'trained_model': model}

# Store results
results = []
class_names = list(le_target.classes_)
print('Evaluation function defined. Ready to train models.')

### 6.1 Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42, max_depth=15, min_samples_split=5)
res_dt = evaluate_model(dt_model, X_train, X_test, y_train, y_test, 'Decision Tree', class_names)
results.append(res_dt)

### 6.2 Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=20,
                                   min_samples_split=5, n_jobs=-1)
res_rf = evaluate_model(rf_model, X_train, X_test, y_train, y_test, 'Random Forest', class_names)
results.append(res_rf)

### 6.3 K-Nearest Neighbors Classifier

KNN is a distance-based model, so we use the **scaled** features to ensure all variables contribute equally.

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=7, weights='distance', n_jobs=-1)
res_knn = evaluate_model(knn_model, X_train_scaled, X_test_scaled, y_train, y_test, 'K-Nearest Neighbors', class_names)
results.append(res_knn)

## 7. Model Comparison

In [ ]:
# Build comparison table
comparison = pd.DataFrame(results)[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]
comparison = comparison.sort_values('F1-Score', ascending=False).reset_index(drop=True)
comparison.style.highlight_max(subset=['Accuracy', 'Precision', 'Recall', 'F1-Score'],
                                color='lightgreen')

In [ ]:
# Visual comparison
comp_melt = comparison.melt(id_vars='Model', var_name='Metric', value_name='Score')

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x='Metric', y='Score', hue='Model', data=comp_melt, palette='Set2', ax=ax)
ax.set_ylim(0, 1.05)
ax.set_title('Model Performance Comparison', fontsize=14)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Model Selection

Based on the comparison above, we select the **best-performing model** by looking at the highest F1-Score (which balances precision and recall). The selected model will be saved for deployment.

**Technical justification:** The model with the highest F1-Score provides the best trade-off between correctly identifying each fertilizer class (recall) and avoiding false recommendations (precision).

**Business justification:** In agriculture, both types of errors matter — recommending the wrong fertilizer wastes resources and can harm soil health, while failing to recommend the right one means missed optimization opportunities. The F1-Score captures both concerns.

In [ ]:
# Select best model
best_row = comparison.iloc[0]
best_name = best_row['Model']
best_obj = [r['trained_model'] for r in results if r['Model'] == best_name][0]
best_uses_scaling = (best_name == 'K-Nearest Neighbors')

print(f'Best model: {best_name}')
print(f'  Accuracy:  {best_row["Accuracy"]:.4f}')
print(f'  Precision: {best_row["Precision"]:.4f}')
print(f'  Recall:    {best_row["Recall"]:.4f}')
print(f'  F1-Score:  {best_row["F1-Score"]:.4f}')
print(f'  Uses scaling: {best_uses_scaling}')

## 8. Feature Importance and Insights

We analyze feature importance from tree-based models to understand which variables drive fertilizer recommendations.

In [ ]:
# Feature importance from Random Forest (most reliable among tree-based models)
importances = rf_model.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('Random Forest Feature Importance:\n')
print(feat_imp.to_string(index=False))

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(x='Importance', y='Feature', data=feat_imp, palette='viridis', ax=ax)
ax.set_title('Feature Importance — Random Forest', fontsize=14)
ax.set_xlabel('Relative Importance')
for i, v in enumerate(feat_imp['Importance']):
    ax.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Decision Tree feature importance for comparison
dt_imp = pd.DataFrame({
    'Feature': feature_cols,
    'DT_Importance': dt_model.feature_importances_,
    'RF_Importance': rf_model.feature_importances_
}).sort_values('RF_Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
x_pos = np.arange(len(dt_imp))
width = 0.35
ax.barh(x_pos - width/2, dt_imp['DT_Importance'], width, label='Decision Tree', color='#66c2a5')
ax.barh(x_pos + width/2, dt_imp['RF_Importance'], width, label='Random Forest', color='#fc8d62')
ax.set_yticks(x_pos)
ax.set_yticklabels(dt_imp['Feature'])
ax.set_xlabel('Importance')
ax.set_title('Feature Importance Comparison: Decision Tree vs Random Forest', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

### Feature Importance Insights — Answering the Research Questions

**RQ1: Which soil and environmental variables influence fertilizer recommendation?**

The feature importance analysis reveals that virtually all input variables contribute to the recommendation. The Random Forest model distributes importance more evenly across features compared to the Decision Tree, which is expected since ensemble methods capture more nuanced interactions. Key soil properties (pH, moisture, organic carbon) and nutrient levels (N, P, K) consistently appear among the influential features, along with agricultural context variables like crop type and region.

**RQ2: Can machine learning models accurately predict the optimal fertilizer?**

Yes. All three models achieved meaningful predictive performance. The comparison table above shows that machine learning can capture the complex relationships between soil, environmental, and agricultural variables to produce reliable fertilizer recommendations.

**RQ3: Which features are the most important drivers?**

The top drivers vary slightly between models, but the most consistently important features tend to include nutrient levels (N, P, K), soil properties, and crop-related variables. This makes agronomic sense — fertilizer recommendations are fundamentally driven by what the soil lacks (nutrient levels), the medium the plant grows in (soil type and properties), and what the plant needs (crop type and growth stage).

## 9. Deployment Preparation

We save the trained model, encoders, scaler, and feature configuration so the model can be reused in a Streamlit application.

In [ ]:
# Save the best model
joblib.dump(best_obj, 'fertilizer_model.pkl')
print(f'Model saved: fertilizer_model.pkl ({best_name})')

# Save preprocessing objects
joblib.dump(label_encoders, 'label_encoders.pkl')
joblib.dump(le_target, 'target_encoder.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(feature_cols, 'feature_columns.pkl')
joblib.dump(best_uses_scaling, 'uses_scaling.pkl')

print('All preprocessing artifacts saved:')
print('  - label_encoders.pkl')
print('  - target_encoder.pkl')
print('  - scaler.pkl')
print('  - feature_columns.pkl')
print('  - uses_scaling.pkl')

### Example Prediction — Simulating a Real User Case

In [ ]:
# Simulate a new sample: a farmer provides the following field data
new_sample = {
    'Soil_Type': 'Clay',
    'Crop_Type': 'Wheat',
    'Crop_Growth_Stage': 'Vegetative',
    'Season': 'Kharif',
    'Irrigation_Type': 'Sprinkler',
    'Previous_Crop': 'Potato',
    'Region': 'South',
    'Soil_pH': 620,
    'Soil_Moisture': 3500,
    'Organic_Carbon': 45,
    'Electrical_Conductivity': 100,
    'Nitrogen_Level': 70,
    'Phosphorus_Level': 40,
    'Potassium_Level': 55,
    'Temperature': 2500,
    'Humidity': 6000,
    'Rainfall': 150000,
    'Fertilizer_Used_Last_Season': 15000,
    'Yield_Last_Season': 400
}

print('Input sample:')
for k, v in new_sample.items():
    print(f'  {k}: {v}')

In [ ]:
# Load saved artifacts and make prediction
model_loaded = joblib.load('fertilizer_model.pkl')
le_dict = joblib.load('label_encoders.pkl')
le_tgt = joblib.load('target_encoder.pkl')
scl = joblib.load('scaler.pkl')
feat_cols = joblib.load('feature_columns.pkl')
needs_scaling = joblib.load('uses_scaling.pkl')

# Encode categorical features
sample_encoded = {}
for col in feat_cols:
    if col in le_dict:
        sample_encoded[col] = le_dict[col].transform([new_sample[col]])[0]
    else:
        sample_encoded[col] = new_sample[col]

# Create DataFrame in correct feature order
sample_df = pd.DataFrame([sample_encoded], columns=feat_cols)

# Scale if necessary
if needs_scaling:
    sample_df = pd.DataFrame(scl.transform(sample_df), columns=feat_cols)

# Predict
prediction = model_loaded.predict(sample_df)
predicted_fertilizer = le_tgt.inverse_transform(prediction)[0]

print(f'\n>>> Recommended Fertilizer: {predicted_fertilizer} <<<')
print(f'\nThis recommendation is ready to be served via a Streamlit app.')

### Deployment Notes

The following artifacts have been saved and are ready for integration into a Streamlit application:

| File | Content |
|---|---|
| `fertilizer_model.pkl` | Trained classification model |
| `label_encoders.pkl` | Encoders for categorical features |
| `target_encoder.pkl` | Encoder for the target variable |
| `scaler.pkl` | StandardScaler fitted on training data |
| `feature_columns.pkl` | Ordered list of feature columns |
| `uses_scaling.pkl` | Flag indicating if model needs scaled input |

The prediction pipeline demonstrated above shows exactly how to load these artifacts, encode a new sample, and produce a recommendation.

## 10. Conclusions

### Key Findings

This project successfully developed a machine learning system for fertilizer recommendation using 10,000 agricultural observations with 19 features and 7 fertilizer classes. The three research questions were addressed:

1. **Influential variables:** Nutrient levels (N, P, K), soil properties (pH, moisture, organic carbon), and agricultural context (crop type, region, season) all contribute meaningfully to fertilizer recommendations. No single variable dominates — the recommendation is inherently multivariate.

2. **Prediction accuracy:** All three models (Decision Tree, Random Forest, KNN) demonstrated that machine learning can predict the optimal fertilizer with meaningful accuracy. The best model was selected based on F1-Score to balance precision and recall across all 7 classes.

3. **Feature importance:** Tree-based feature importance analysis confirmed that the most important drivers align with agronomic knowledge — what the soil contains, what the plant needs, and the environmental context.

### Business Value

This system can help:
- **Farmers** make data-driven fertilizer decisions without requiring deep agronomic expertise.
- **Agricultural companies** (e.g., Fertinagro Biotech) offer advisory services backed by data.
- **Sustainability goals** by reducing over-fertilization and nutrient runoff.

### Limitations

- The dataset appears to use integer-encoded values for some variables (e.g., pH values in the 45–849 range rather than typical 0–14), which may reflect a specific data collection methodology or scaling.
- The model was trained on a single dataset and should be validated on external data before production deployment.
- Label encoding for categorical variables introduces an implicit ordinal relationship that may not exist. One-hot encoding could be explored as an alternative.

### Future Improvements

- Test additional models (Gradient Boosting, XGBoost, Neural Networks).
- Perform hyperparameter tuning using cross-validation.
- Explore one-hot encoding vs. label encoding impact.
- Validate on real-world field data.
- Deploy the Streamlit application for end-user interaction.

---

*This notebook was developed as part of the AI for Business Analytics course at Purdue University, following the PDID (Problem–Data–Insights–Deployment) framework.*